# Model Derivation: symbolic checks of the closed-form algebra

This notebook verifies, with SymPy, the identities behind the closed-form results of *Investing in Artificial General Intelligence*, and cross-checks some of them against the model code. It follows the model of record of the paper: the single firm solves the exact piecewise L-regime stopping problem (Proposition 1), credit uses the recovery rule `@eq-recovery`, and the duopoly is the preemption game in the exact primitives (Proposition 3). The symbolic functions live in `src/ai_lab_investment/models/symbolic_duopoly.py`, and `tests/test_symbolic_duopoly.py` asserts every residual shown here.

**What a cell establishes.** A symbolic residual that simplifies to zero verifies that one identity, for symbolic parameters, under the sign assumptions stated in the cell. It does not establish that a candidate value function is optimal, that a trigger exists or is unique, or any equilibrium level. Numerical cells evaluate a closed form at the baseline and compare it with the model code; they establish agreement at the baseline only. Each section ends with a line saying what it establishes.

**What is outside the notebook.** The optimal-stopping arguments, uniqueness of the capacity maximum in Proposition 1(iii), the waiting-margin criterion as a criterion, the existence of the rent-equalization trigger $X_P$ and every result of Proposition 3(iii) to (v), the order of $X_L$ and $X_P$ and which of them dates the first investment, the coupled default boundary and the levered claims of the game, Remark 4, Dario's dilemma, and the installed-value concavity of issue #181 (pending). The completeness audit at the end lists them.

**Outline:**
1. Setup and notation
2. Characteristic equations
3. The post-switch problem (Proposition 1(ii))
4. The L-regime problem in the model of record (Proposition 1(iii), the pure-power approximation, the waiting margin)
5. The effective revenue coefficient and the allocation (Proposition 1(i))
6. The default boundary and faith-based survival (Proposition 2, `@eq-phi-tilde`)
7. The recovery rule (`@eq-recovery`)
8. The contest and the follower (Proposition 3(i) and (ii))
9. The Green's function of the L-regime operator (`@eq-green`)
10. Parameter admissibility
11. Summary
12. Completeness audit

**References:** Guo, Miao and Morellec (2005); Dixit and Pindyck (1994), Chapter 6; Leland (1994).

In [1]:
import numpy as np
import sympy as sp
from IPython.display import Math
from IPython.display import display as ipy_display
from scipy import optimize

import ai_lab_investment.models.symbolic_duopoly as sd
from ai_lab_investment.models.base_model import SingleFirmModel
from ai_lab_investment.models.duopoly import DuopolyModel
from ai_lab_investment.models.parameters import ModelParameters
from ai_lab_investment.models.piecewise_option import (
    PiecewiseOptionModel,
    reduced_form_reference,
)
from ai_lab_investment.models.preemption_exact import ExactPreemptionGame, ForcedClaim

sp.init_printing(use_latex="mathjax")


def show(label: str, expr):
    # Display a labeled equation.
    ipy_display(Math(label + r" \;:\quad " + sp.latex(expr)))


def check_zero(name: str, residual):
    # Print a symbolic residual and fail loudly if it is not zero.
    print(f"{name}: residual = {residual}")
    if residual != 0:
        msg = f"{name}: residual does not simplify to zero"
        raise ValueError(msg)


syms = sd.define_symbols()
params = ModelParameters()
p = params

## 1. Setup and notation

Demand follows a regime-switching geometric Brownian motion, $dX_t/X_t = \mu_s\,dt + \sigma\,dW_t$, $s \in \{L, H\}$, with a common volatility. Regime $L$ switches to the absorbing regime $H$ at Poisson rate $\lambda$. The firm installs capacity $K$ once, at cost $I(K) = cK^\gamma$ plus the operating-cost perpetuity $\delta K/r$, and allocates the fraction $\phi$ to training and $1-\phi$ to inference.

In [2]:
print("Baseline parameters:")
print(f"  r={p.r}, mu_L={p.mu_L}, mu_H={p.mu_H}, sigma={p.sigma}")
print(f"  lambda={p.lam}, alpha={p.alpha}, gamma={p.gamma}, c={p.c}, delta={p.delta}")
print(f"  beta_H={p.beta_H:.6f}, beta_L^+={p.beta_L:.6f}")

Baseline parameters:
  r=0.12, mu_L=0.01, mu_H=0.06, sigma=0.25
  lambda=0.1, alpha=0.4, gamma=1.5, c=1.0, delta=0.03
  beta_H=1.552859, beta_L^+=3.014995


## 2. Characteristic equations

A power $X^\beta$ solves the homogeneous H-regime equation iff $Q_H(\beta) = \tfrac12\sigma^2\beta(\beta-1) + \mu_H\beta - r = 0$, and the homogeneous L-regime equation, in which the switch acts as an extra discount, iff $Q_L(\beta) = \tfrac12\sigma^2\beta(\beta-1) + \mu_L\beta - (r+\lambda) = 0$.

In [3]:
ch_H = sd.characteristic_equation_H(syms)
ch_L = sd.characteristic_equation_L(syms)
show("Q_H(\\beta)", ch_H["equation"])
show("\\beta_H^+", ch_H["positive_root_expr"])
show("Q_L(\\beta)", ch_L["equation"])
show("\\beta_L^+", ch_L["positive_root_expr"])

subs_L = {
    syms["sigma"]: p.sigma,
    syms["mu_L"]: p.mu_L,
    syms["r"]: p.r,
    syms["lam"]: p.lam,
}
subs_H = {syms["sigma"]: p.sigma, syms["mu_H"]: p.mu_H, syms["r"]: p.r}
bH = float(ch_H["positive_root_expr"].subs(subs_H))
bLp = float(ch_L["positive_root_expr"].subs(subs_L))
bLn = float(ch_L["negative_root_expr"].subs(subs_L))
print(f"beta_H = {bH:.6f} (code {p.beta_H:.6f})")
print(f"beta_L^+ = {bLp:.6f} (code {p.beta_L:.6f}), beta_L^- = {bLn:.6f}")
ordered = bLn < 0 < 1 < bH < bLp
print(f"beta_L^- < 0 < 1 < beta_H < beta_L^+ at baseline: {ordered}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

beta_H = 1.552859 (code 1.552859)
beta_L^+ = 3.014995 (code 3.014995), beta_L^- = -2.334995
beta_L^- < 0 < 1 < beta_H < beta_L^+ at baseline: True


Evaluating $Q_L$ at the H-regime root uses $Q_H(\beta_H) = 0$: $Q_L(\beta_H) = (\mu_L - \mu_H)\beta_H - \lambda$, negative since $\mu_H > \mu_L$. This is the identity behind $\Gamma > 0$ in `@eq-particular-C` and, because $Q_L$ is an upward parabola, behind $\beta_H < \beta_L^+$ (remark on the pure-power form in Internet Appendix A).

In [4]:
l_ode = sd.l_regime_ode(syms)
show("Q_L(\\beta_H) \\text{ given } Q_H(\\beta_H)=0", l_ode["Q_L_at_beta_H_reduced"])
check_zero(
    "Q_L(beta_H) - Q_H(beta_H) - [(mu_L - mu_H) beta_H - lambda]",
    l_ode["Q_L_reduction_residual"],
)

<IPython.core.display.Math object>

Q_L(beta_H) - Q_H(beta_H) - [(mu_L - mu_H) beta_H - lambda]: residual = 0


*Established:* the root formulas, their baseline values, and the reduction of $Q_L(\beta_H)$ (symbolic). The general ordering $\beta_L^- < 0$, $\beta_H > 1$, $\beta_L^+ > \beta_H$ is checked here at baseline only; it is proved in the Lean package.

## 3. The post-switch problem (Proposition 1(ii))

After the switch the firm trains with all its capacity ($\phi = 1$), since inference earns nothing in regime $H$. With exercise payoff $aK^\alpha X - \Theta(K)$, $\Theta(K) = cK^\gamma + \delta K/r$, and continuation value $B X^{\beta_H}$, value matching and smooth pasting give the trigger with markup $\beta_H/(\beta_H-1)$ (Step 1 of the Proof of Proposition 1).

In [5]:
h_res = sd.h_regime_option_value(syms)
show("X_H^*(K)", sp.simplify(h_res["trigger"]))
ratio = sp.simplify(h_res["trigger"] * h_res["revenue_coeff"] / h_res["total_cost"])
check_zero(
    "trigger * revenue / cost - beta_H/(beta_H - 1)",
    sp.simplify(ratio - syms["beta_H"] / (syms["beta_H"] - 1)),
)

<IPython.core.display.Math object>

trigger * revenue / cost - beta_H/(beta_H - 1): residual = 0


The option coefficient is proportional to $h(K) = (aK^\alpha)^{\beta_H}/\Theta(K)^{\beta_H - 1}$ (Step 3). Its first-order condition `@eq-foc-K` does not involve $a$, and is linear in $u = K^{\gamma-1}$; its root is the closed form `@eq-K-H`. The exponents of $h$ as $K \to 0$ and $K \to \infty$ are positive and negative exactly under the upper and lower bounds of (A2), which is the global-maximum argument of Step 4.

In [6]:
cap = sd.post_switch_capacity_closed_form(syms)
show("\\partial \\ln h / \\partial K", cap["foc"])
print("Revenue coefficient a drops out of the condition:", cap["a_drops_out"])
show("K_H^{*\\,\\gamma - 1}", cap["K_H_power_gamma_minus_1"])
check_zero("root in u minus closed form", cap["foc_residual_at_K_H"])
show("\\text{exponent of } h \\text{ as } K \\to 0", cap["exponent_small_K"])
show("\\text{exponent of } h \\text{ as } K \\to \\infty", cap["exponent_large_K"])

piece = PiecewiseOptionModel(p)
subs_all = {
    syms["r"]: p.r,
    syms["mu_L"]: p.mu_L,
    syms["mu_H"]: p.mu_H,
    syms["sigma"]: p.sigma,
    syms["lam"]: p.lam,
    syms["alpha"]: p.alpha,
    syms["gamma"]: p.gamma,
    syms["c"]: p.c,
    syms["delta"]: p.delta,
    syms["beta_H"]: p.beta_H,
    syms["beta_L_plus"]: p.beta_L,
}
K_H_closed = float(cap["K_H"].subs(subs_all))
Theta_H = p.c * K_H_closed**p.gamma + p.delta * K_H_closed / p.r
X_H_closed = p.beta_H / (p.beta_H - 1) * Theta_H / (p.A_H * K_H_closed**p.alpha)
print(f"\nK_H* closed form = {K_H_closed:.8f}; code = {piece.K_H:.8f}")
print(f"X_H* closed form = {X_H_closed:.8f}; code = {piece.X_H:.8f}")
e_small = float(cap["exponent_small_K"].subs(subs_all))
e_large = float(cap["exponent_large_K"].subs(subs_all))
print(f"Exponents at baseline: small K {e_small:+.4f}, large K {e_large:+.4f}")

<IPython.core.display.Math object>

Revenue coefficient a drops out of the condition: True


<IPython.core.display.Math object>

root in u minus closed form: residual = 0


<IPython.core.display.Math object>

<IPython.core.display.Math object>


K_H* closed form = 0.00672664; code = 0.00672664
X_H* closed form = 0.00278296; code = 0.00278296
Exponents at baseline: small K +0.0683, large K -0.2081


*Established:* the trigger formula, the closed form `@eq-K-H` as the root of `@eq-foc-K`, and the independence of the root from the revenue coefficient (symbolic); the signs of the limiting exponents at baseline, and agreement of the closed form with the code (numerical; the code's optimizer agrees to about six digits).

## 4. The L-regime problem in the model of record

The value $F_L$ of a firm that has not invested solves `@eq-hjb-L`,
$$\tfrac12\sigma^2X^2F_L'' + \mu_L XF_L' + \lambda[F_H(X) - F_L(X)] - rF_L = 0,$$
where $F_H$ is what the firm holds if the switch arrives first. Below the post-switch trigger $X_H^*$ that is the option $B_H X^{\beta_H}$; at or above it the firm invests at once, so the forcing is the exercised payoff. The forcing is therefore piecewise, and so is the solution.

### 4.1 Below $X_H^*$

The particular solution is $\Gamma X^{\beta_H}$ with $\Gamma = -\lambda B_H/Q_L(\beta_H)$ (`@eq-particular-C`); the condition $F_L(0) = 0$ removes $X^{\beta_L^-}$, so $F_L = A_1 X^{\beta_L^+} + \Gamma X^{\beta_H}$ on $(0, \min\{X_H^*, X^*\})$ (`@eq-option-L-general`).

In [7]:
show("\\Gamma", l_ode["C"])
check_zero("ODE residual of Gamma X^beta_H below X_H*", l_ode["ode_residual"])
print("Scope:", l_ode["note"])

<IPython.core.display.Math object>

ODE residual of Gamma X^beta_H below X_H*: residual = 0
Scope: Valid below the post-switch trigger X_H* only. Setting A1 = 0 and extending C * X^{beta_H}'s functional form to the whole continuation region is the pure-power approximation, not a solution of the HJB (see `pure_power_smooth_fit`).


### 4.2 Above $X_H^*$, post-switch scale precommitted to $K_H^*$

The forcing is $\lambda(p_H X - q_H)$ with $p_H = A_H(K_H^*)^\alpha$ and $q_H = \delta K_H^*/r + I(K_H^*)$, and the affine particular solution is $gX + k$, $g = \lambda p_H/(r+\lambda-\mu_L)$, $k = -\lambda q_H/(r+\lambda)$, to which $G_1X^{\beta_L^+} + G_2X^{\beta_L^-}$ is added (`@eq-option-L-upper`).

In [8]:
up = sd.l_regime_upper_particular(syms)
show("g", up["g"])
show("k", up["k"])
check_zero("ODE residual of g X + k above X_H*", up["residual"])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

ODE residual of g X + k above X_H*: residual = 0


### 4.3 The two-region system (Proposition 1(iii))

Value and slope matching at $X_H^*$ and value matching and smooth pasting at $X^*$ against the payoff $aX - b$, with $a = A_{\text{eff}}(\phi^*, K)$ and $b = \delta K/r + I(K)$, are four conditions. Three of them are linear in $(A_1, G_1, G_2)$ given $X^*$; substituting leaves one scalar equation in $X^*$ at a given capacity, which the code solves by Brent's method.

In [9]:
two = sd.two_region_conditions(syms)
for name, cond in two["conditions"].items():
    print(name)
    ipy_display(cond)
names = two["symbols"]
free = [k for k in ("A1", "G1", "G2") if two["scalar_equation_in_X_star"].has(names[k])]
print("Coefficients solved from three conditions:", list(two["coefficients"]))
print("Remaining condition involves only X*, X_H* and data:", not free)

value_matching_X_H

         β_L_plus           β_H            β_L_plus            β_L_minus       ↪
A₁⋅X_H__*         + Γ⋅X_H__*    = G₁⋅X_H__*         + G₂⋅X_H__*          + X_H ↪

↪          
↪ __*⋅g + k

slope_matching_X_H


         β_L_plus - 1                    β_H - 1                β_L_plus - 1   ↪
A₁⋅X_H__*            ⋅β_L_plus + Γ⋅X_H__*       ⋅β_H = G₁⋅X_H__*            ⋅β ↪

↪                    β_L_minus - 1              
↪ _L_plus + G₂⋅X_H__*             ⋅β_L_minus + g

value_matching_X_star


       β_L_plus          β_L_minus                          
G₁⋅X__*         + G₂⋅X__*          + X__*⋅g + k = X__*⋅a - b

smooth_pasting_X_star


       β_L_plus - 1                   β_L_minus - 1                  
G₁⋅X__*            ⋅β_L_plus + G₂⋅X__*             ⋅β_L_minus + g = a

Coefficients solved from three conditions: [A_1, G_1, G_2]
Remaining condition involves only X*, X_H* and data: True


### 4.4 The exercise premium (Step 7) and the scale condition

Let $S$ be the value of never investing before the switch. The policy "install $(K, \phi^*)$ when demand first reaches $\bar X$" is worth $F_L(X) = S(X) + [P(\bar X; K) - S(\bar X)](X/\bar X)^{\beta_L^+}$ below $\bar X$ (`@eq-premium`). The added term solves the homogeneous equation because $Q_L(\beta_L^+) = 0$, and the first-order condition of the coefficient $[P - S]\bar X^{-\beta_L^+}$ in $\bar X$ is smooth pasting against the payoff. For generic functions $P$ and $S$:

In [10]:
prem = sd.premium_representation(syms)
show("\\text{coefficient}", prem["coefficient"])
show("\\partial_{\\bar X}\\,\\text{coefficient}", prem["foc_in_Xbar"])
show("F_L'(\\bar X) - P'(\\bar X)", prem["smooth_pasting"])
check_zero(
    "added term minus Q_L(beta_L^+) times itself (homogeneous equation)",
    prem["homogeneous_residual"],
)
check_zero(
    "smooth pasting + Xbar^{beta_L^+} * first-order condition",
    prem["smooth_pasting_minus_foc"],
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

added term minus Q_L(beta_L^+) times itself (homogeneous equation): residual = 0
smooth pasting + Xbar^{beta_L^+} * first-order condition: residual = 0


Numerically, the code's precommitted-scale optimum (`PiecewiseOptionModel.precommitted_optimum`) satisfies the four matching conditions, the scale condition `@eq-scale-foc`, and the decomposition $A_1 = A_w + [P - S]\bar X^{-\beta_L^+}$, $G_1 = [P - S]\bar X^{-\beta_L^+}$, $G_2 = G_w$ of Step 7, where $(A_w, G_w)$ are the coefficients of $S$.

In [11]:
opt = piece.precommitted_optimum()
A_w, G_w = piece.pure_switching_coeffs()
bp, bn, h = opt.beta_L_pos, opt.beta_L_neg, opt.beta_H
X_H, X_s, K_s = opt.X_H, opt.X_star, opt.K


def upper(x):
    return opt.D_1 * x**bp + opt.D_2 * x**bn + opt.g * x + opt.k


def upper_slope(x):
    return bp * opt.D_1 * x ** (bp - 1) + bn * opt.D_2 * x ** (bn - 1) + opt.g


lower = opt.A_1 * X_H**bp + opt.C * X_H**h
lower_slope = bp * opt.A_1 * X_H ** (bp - 1) + h * opt.C * X_H ** (h - 1)
print(f"Policy: X* = {X_s:.6f}, K* = {K_s:.6f}, phi* = {opt.phi:.6f}")
print(f"X_H* = {X_H:.6f}; two-region: {opt.two_region}")
print(f"  value matching at X_H*:  {lower - upper(X_H):+.2e}")
print(f"  slope matching at X_H*:  {lower_slope - upper_slope(X_H):+.2e}")
print(f"  value matching at X*:    {upper(X_s) - (opt.a_L * X_s - opt.b_L):+.2e}")
print(f"  smooth pasting at X*:    {upper_slope(X_s) - opt.a_L:+.2e}")
lhs = p.alpha * opt.a_L * X_s
rhs = K_s * (p.delta / p.r + p.c * p.gamma * K_s ** (p.gamma - 1))
print(f"  scale condition (eq-scale-foc), relative gap: {lhs / rhs - 1:+.2e}")
premium = opt.a_L * X_s - opt.b_L - piece.pure_switching_value(X_s)
print(f"\nA_w = {A_w:.4e}, G_w = {G_w:.4e}")
A_1_step7 = A_w + premium * X_s ** (-bp)
print(f"A_1 = {opt.A_1:.4e}; A_w + [P - S] X*^(-beta_L^+) = {A_1_step7:.4e}")
print(f"G_1 = {opt.D_1:.4e}; [P - S] X*^(-beta_L^+) = {premium * X_s ** (-bp):.4e}")
print(f"G_2 = {opt.D_2:.4e}; G_w = {G_w:.4e}")
at_KH = piece.solve(piece.K_H, opt.phi)
if at_KH is None:
    msg = "No finite trigger at K_H*"
    raise RuntimeError(msg)
print(f"\nA_1 at the capacity K_H* of the approximation = {at_KH.A_1:.4e}")
print(f"(trigger at that capacity {at_KH.X_star:.6f})")
S_XH = piece.pure_switching_value(X_H)
gap_S = S_XH / (opt.C * X_H**h) - 1
print(f"S(X_H*) / (Gamma X_H*^beta_H) - 1 = {gap_S:+.2%} (A_w < 0)")

Policy: X* = 0.009258, K* = 0.015784, phi* = 0.700856
X_H* = 0.002783; two-region: True
  value matching at X_H*:  +0.00e+00
  slope matching at X_H*:  -1.75e-14
  value matching at X*:    +0.00e+00
  smooth pasting at X*:    +0.00e+00
  scale condition (eq-scale-foc), relative gap: -1.44e-09

A_w = -1.1834e+04, G_w = 7.3915e-11
A_1 = -8.5285e+03; A_w + [P - S] X*^(-beta_L^+) = -8.5285e+03
G_1 = 3.3056e+03; [P - S] X*^(-beta_L^+) = 3.3056e+03
G_2 = 7.3915e-11; G_w = 7.3915e-11

A_1 at the capacity K_H* of the approximation = -1.0070e+04
(trigger at that capacity 0.007220)
S(X_H*) / (Gamma X_H*^beta_H) - 1 = -10.27% (A_w < 0)


*Established:* the particular solutions on both sides of $X_H^*$, the linear structure of the two-region system, and the two identities of Step 7 (symbolic); that the code's optimum satisfies the conditions of Proposition 1(iii), and the values of $A_w$ and $A_1$ quoted in Internet Appendix A (numerical, at baseline). Not established: that the capacity search has a single maximum (a grid finding in the code), or that the policy is optimal among all stopping rules.

### 4.5 What the L-regime insufficiency condition says

The condition `@eq-l-insufficiency`, $\Phi_L = (1 - 1/\beta_L^+)/\alpha \geq 1$ (the former Assumption A3), says that the standalone L-regime scale problem has no interior optimal capacity. It is a description: it does not make $A_1$ vanish, and the exact solution above has $A_1 < 0$. Earlier versions of this notebook stated that $\Phi_L \geq 1$ implies $A_1 = 0$ and that the single power is therefore valid at baseline; that statement was wrong and is withdrawn.

In [12]:
ins = sd.l_regime_insufficiency(syms)
show("\\Phi_L", ins["option_premium_ratio_L"])
status = sd.verify_pure_power_approximation(p)
Phi_L = status["option_premium_ratio_L"]
print(f"Phi_L = {Phi_L:.4f} at baseline (condition holds: {Phi_L >= 1})")
print(f"A_1 of the exact precommitted-scale solution: {status['A_1_exact']:.4e}")
print(f"Single power exact: {status['pure_power_form_exact']}")
print(status["conclusion"])

<IPython.core.display.Math object>

Phi_L = 1.6708 at baseline (condition holds: True)
A_1 of the exact precommitted-scale solution: -8.5285e+03
Single power exact: False
Phi_L = 1.6708 describes the standalone L-regime scale problem and says nothing about A_1. The exact precommitted-scale solution has A_1 = -8529, and the smooth-fit coefficient 16.52 differs from Gamma = 21.14, so F_L = B X^{beta_H} is an approximation, with its error reported by piecewise_option.py.


### 4.6 The pure-power approximation

Imposing a single power $BX^{\beta_H}$ on the whole continuation region and pinning $B$ by value matching and smooth pasting gives the trigger `@eq-trigger-phi`, $X^{\text{pp}} = [\beta_H/(\beta_H-1)]\,b/a$, and the smooth-fit coefficient $B = (aX^{\text{pp}} - b)/(X^{\text{pp}})^{\beta_H}$. Substituted in the HJB below $X_H^*$, $BX^{\beta_H}$ leaves the residual $Q_L(\beta_H)(B - \Gamma)X^{\beta_H}$, which vanishes only if $B = \Gamma$.

In [13]:
pp = sd.pure_power_smooth_fit(syms)
show("X^{\\text{pp}}", pp["trigger"])
show("B", pp["smooth_fit_coeff"])
show("\\text{HJB residual of } BX^{\\beta_H}", pp["hjb_residual_factored"])
check_zero(
    "HJB residual minus Q_L(beta_H)(B - Gamma) X^beta_H", pp["factorization_residual"]
)

ref = reduced_form_reference(p)
B_val, Gamma_val = ref["smooth_fit_coeff"], ref["C"]
print(f"\nAt baseline: smooth-fit B = {B_val:.4f}, Gamma = {Gamma_val:.4f}")
print(
    f"X^pp = {ref['X_star']:.6f}, K^pp = {ref['K_star']:.6f} (= K_H* {piece.K_H:.6f})"
)
g_star = ((1 - opt.phi) ** p.alpha + p.lam * p.A_H * opt.phi**p.alpha) / (
    p.r - p.mu_L + p.lam
)
print(f"X_H* A_H / g(phi*) = {piece.X_H * p.A_H / g_star:.6f} (Step 4)")
dX, dK = ref["X_star"] / X_s - 1, ref["K_star"] / K_s - 1
print(f"Against the precommitted-scale policy: trigger {dX:+.1%}, capacity {dK:+.1%}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

HJB residual minus Q_L(beta_H)(B - Gamma) X^beta_H: residual = 0

At baseline: smooth-fit B = 16.5246, Gamma = 21.1405
X^pp = 0.004722, K^pp = 0.006727 (= K_H* 0.006727)
X_H* A_H / g(phi*) = 0.004722 (Step 4)
Against the precommitted-scale policy: trigger -49.0%, capacity -57.4%


*Established:* the trigger and coefficient of the approximation, and the factorization of its HJB residual (symbolic); $B \neq \Gamma$ at baseline, so the single power solves the HJB on no interval (numerical). The value loss of the approximation is computed in `piecewise_option.py` and reported in Internet Appendix B, not here.

### 4.7 Scale chosen at the switch: the waiting margin

When the firm chooses its scale on arrival of the switch, the post-switch payoff is the envelope $\max_K[A_HK^\alpha X - \delta K/r - cK^\gamma]$. Dropping the operating cost, which is of lower order in $X$, the envelope of $aX$ is $E(a)X^\theta$ with $\theta = \gamma/(\gamma-\alpha)$; the waiting value forced by $\lambda E(A_H)X^\theta$ has the particular solution $\lambda E(A_H)X^\theta/(-Q_L(\theta))$; and the best pre-switch payoff is $E(g(\phi^*))X^\theta$. The ratio of the leading coefficients is the waiting margin `@eq-waiting-margin`.

In [14]:
wm = sd.waiting_margin_asymptotic(syms)
show("\\theta", wm["theta"])
show("E(a)", wm["envelope_coefficient"])
check_zero(
    "first-order condition, K^(gamma-alpha) = alpha a X/(c gamma)",
    wm["envelope_foc_residual"],
)
check_zero(
    "envelope value minus (1 - alpha/gamma) a X K^alpha", wm["envelope_value_residual"]
)
check_zero("d/dX log E(a) (E does not depend on X)", wm["growth_residual"])
check_zero(
    "log ratio of leading terms minus log of eq-waiting-margin", wm["margin_residual"]
)

subs_wm = dict(subs_all)
subs_wm.update({
    sp.Symbol("A_H", positive=True): p.A_H,
    sp.Symbol("g", positive=True): g_star,
})
theta_val = float(wm["theta"].subs(subs_wm))
margin_val = float(wm["margin"].subs(subs_wm))
sol = SingleFirmModel(p).solve()
Q_theta = float(wm["Q_L_theta"].subs(subs_wm))
print(
    f"\ntheta = {theta_val:.4f} < beta_L^+ = {p.beta_L:.4f}; Q_L(theta) = {Q_theta:.4f}"
)
print(f"Waiting margin: formula {margin_val:.6f}, code {sol.waiting_margin:.6f}")
print(f"Code: invests before the switch at baseline: {sol.invests_before_switch}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

first-order condition, K^(gamma-alpha) = alpha a X/(c gamma): residual = 0
envelope value minus (1 - alpha/gamma) a X K^alpha: residual = 0
d/dX log E(a) (E does not depend on X): residual = 0
log ratio of leading terms minus log of eq-waiting-margin: residual = 0

theta = 1.3636 < beta_L^+ = 3.0150; Q_L(theta) = -0.1909
Waiting margin: formula 1.077342, code 1.077342
Code: invests before the switch at baseline: False


*Established:* that the ratio of the leading asymptotic terms of the two values is the expression `@eq-waiting-margin` (symbolic), and its baseline value (numerical). Not established: that the margin decides whether a finite pre-switch trigger exists. The paper labels that criterion as derived from growth rates and confirmed by the linear complementarity solve, not proved; the waiting result itself comes from the code.

## 5. The effective revenue coefficient and the allocation (Proposition 1(i))

Installed value in regime $L$ is $A_{\text{eff}}(\phi, K)X - \delta K/r$ with
$$A_{\text{eff}}(\phi, K) = \frac{[(1-\phi)K]^\alpha}{r - \mu_L + \lambda} + \frac{\lambda}{r - \mu_L + \lambda}\cdot\frac{(\phi K)^\alpha}{r - \mu_H}$$
(`@eq-a-eff`). The pre-switch payoff is $A_{\text{eff}}X - \Theta(K)$, and neither $\Theta$ nor the continuation value of a firm that has not invested depends on $\phi$, so at every capacity and every trigger the allocation maximizes $A_{\text{eff}}$. That is why the same $\phi^*$ appears in the exact problem and in the approximation.

In [15]:
aeff = sd.effective_revenue_coefficient(syms)
show("A_{\\text{eff}}", aeff["A_eff"])
show("\\partial A_{\\text{eff}}/\\partial\\phi", aeff["dA_dphi"])
show("\\partial A_{\\text{eff}}/\\partial\\lambda", aeff["dA_dlam_quotient_form"])
check_zero(
    "dA_eff/dlambda minus the quotient form of the Proof of Proposition 2",
    aeff["dA_dlam_residual"],
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

dA_eff/dlambda minus the quotient form of the Proof of Proposition 2: residual = 0


Write $A_{\text{eff}} = g(\phi)K^\alpha$ and drop the positive factor $1/(r-\mu_L+\lambda)$. With $\rho = (\lambda A_H)^{1/(1-\alpha)}$, the first-order condition vanishes at $\phi^* = \rho/(1+\rho)$ (`@eq-phi-star`), $g$ is strictly concave on $(0,1)$, and at $\phi^*$ the inference term is the fraction $1-\phi^*$ of $g$ (used in Section 7).

In [16]:
ps = sd.phi_star_closed_form(syms)
show("\\phi^*", ps["phi_star"])
check_zero("dg/dphi at phi = rho/(1 + rho)", ps["foc_residual"])
show("g''(\\phi)", ps["second_derivative"])
check_zero(
    "g'' minus alpha(alpha - 1)[(1-phi)^(alpha-2) + lambda A_H phi^(alpha-2)]",
    ps["second_derivative_residual"],
)
check_zero(
    "inference share of g at phi* minus (1 - phi*)", ps["inference_share_residual"]
)
show("\\partial\\phi^*/\\partial\\rho", ps["dphi_drho"])
show("\\partial\\rho/\\partial\\lambda", ps["drho_dlam"])
show("\\partial\\rho/\\partial\\mu_H", ps["drho_dmu_H"])
show("\\partial\\rho/\\partial\\mu_L", ps["drho_dmu_L"])

<IPython.core.display.Math object>

dg/dphi at phi = rho/(1 + rho): residual = 0


<IPython.core.display.Math object>

g'' minus alpha(alpha - 1)[(1-phi)^(alpha-2) + lambda A_H phi^(alpha-2)]: residual = 0
inference share of g at phi* minus (1 - phi*): residual = 0


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The second derivative is negative because $\alpha(\alpha-1) < 0$ and both bracketed terms are positive; $\phi^*$ increases in $\rho$, and $\rho$ increases in $\lambda$ and in $\mu_H$ (for $r > \mu_H$) and does not depend on $\mu_L$. The Inada behaviour at the two ends is checked numerically:

In [17]:
base_subs = {
    syms["r"]: p.r,
    syms["mu_L"]: p.mu_L,
    syms["mu_H"]: p.mu_H,
    syms["lam"]: p.lam,
    syms["alpha"]: p.alpha,
    syms["K"]: 1.0,
}
dA = sp.lambdify(syms["phi"], aeff["dA_dphi"].subs(base_subs), "numpy")
for eps in (1e-2, 1e-4, 1e-6):
    lo_val, hi_val = float(dA(eps)), float(dA(1 - eps))
    print(f"  phi = {eps:.0e}: {lo_val:+.3e};  1 - phi = {eps:.0e}: {hi_val:+.3e}")

phi_closed = float(ps["phi_star"].subs(base_subs))
phi_code = SingleFirmModel(p).closed_form_phi_star()
print(f"\nphi* closed form = {phi_closed:.8f}; code = {phi_code:.8f}")
print(f"precommitted-scale optimum uses {opt.phi:.8f}")
print(f"rho = {float(ps['rho'].subs(base_subs)):.4f}")
lams = (0.03, 0.05, 0.10, 0.20, 0.50)
phis = [float(ps["phi_star"].subs({**base_subs, syms["lam"]: x})) for x in lams]
print(
    "phi*(lambda):", ", ".join(f"{x}: {v:.4f}" for x, v in zip(lams, phis, strict=True))
)

  phi = 1e-02: +4.840e+01;  1 - phi = 1e-02: -2.699e+01
  phi = 1e-04: +7.955e+02;  1 - phi = 1e-04: -4.753e+02
  phi = 1e-06: +1.264e+04;  1 - phi = 1e-06: -7.580e+03

phi* closed form = 0.70085572; code = 0.70085572
precommitted-scale optimum uses 0.70085572
rho = 2.3429
phi*(lambda): 0.03: 0.2395, 0.05: 0.4246, 0.1: 0.7009, 0.2: 0.8815, 0.5: 0.9716


*Established:* the closed form, strict concavity, the inference-share identity and the signs of the comparative statics (symbolic); the Inada limits and agreement with the code (numerical, at baseline).

## 6. The default boundary and faith-based survival (Proposition 2)

Under the single-boundary reduction, $X_D = M\,(C_D/r + \delta K/r)/A_{\text{eff}}$ with $M = \beta_L^-/(\beta_L^- - 1) \in (0,1)$ (`@eq-default-boundary`). At a zero coupon it is the shutdown boundary. Treating $M$ and $A_{\text{eff}}$ as separate symbols gives the partial effects:

In [18]:
db = sd.default_boundary_derivation(syms)
show("X_D", db["X_D"])
show("\\partial X_D/\\partial A_{\\text{eff}}", db["dXD_dA_eff"])
show("\\partial X_D/\\partial C_D", db["dXD_dcD"])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The arrival rate moves $X_D$ through both. The $A_{\text{eff}}$ channel lowers $X_D$ iff $\partial A_{\text{eff}}/\partial\lambda > 0$, which with symmetric shares holds iff $\phi > \underline\phi = \Omega/(1+\Omega)$, $\Omega = [(r-\mu_H)/(r-\mu_L)]^{1/\alpha}$ (`@eq-phi-underbar`). The markup channel raises it: $\beta_L^-$ falls with $\lambda$ and pushes $M$ towards one. With $\mathcal R = \sqrt{(\mu_L - \sigma^2/2)^2 + 2\sigma^2(r+\lambda)}$:

In [19]:
mk = sd.markup_channel(syms)
show("\\beta_L^-", mk["beta_neg"])
check_zero("Q_L(beta_L^-)", mk["root_residual"])
check_zero("dM/dlambda - 1/[(beta_L^- - 1)^2 R]", mk["dM_dlam_residual"])
check_zero("m - 1/[beta_L^- (beta_L^- - 1) R]", mk["m_residual"])

<IPython.core.display.Math object>

Q_L(beta_L^-): residual = 0
dM/dlambda - 1/[(beta_L^- - 1)^2 R]: residual = 0
m - 1/[beta_L^- (beta_L^- - 1) R]: residual = 0


In logs, $\partial\ln X_D/\partial\lambda = m - \partial\ln A_{\text{eff}}/\partial\lambda$. With symmetric shares both channels depend on $\phi$ only through the odds $\zeta = (\phi/(1-\phi))^\alpha$, and the net condition $\partial X_D/\partial\lambda < 0$ is linear in $\zeta$. It holds iff $\zeta > (r-\mu_H)(1+m\Delta)/[(r-\mu_L) - m\lambda\Delta]$, $\Delta = r-\mu_L+\lambda$, that is iff $\phi > \tilde\phi$ of `@eq-phi-tilde`, given the regularity condition $(r-\mu_L) > m\lambda\Delta$.

In [20]:
ft = sd.exact_faith_threshold(syms)
show("\\underline\\phi", ft["phi_underbar"])
show("\\tilde\\phi", ft["phi_tilde"])
show("\\text{net condition}/\\Pi_L \\text{ in the odds } \\zeta", ft["net_in_odds"])
check_zero("d ln A_eff/d lambda minus the closed form", ft["dlnA_dlam_residual"])
check_zero(
    "net condition / Pi_L minus its form in the odds", ft["linear_in_odds_residual"]
)
check_zero("odds of W^(1/alpha)/(1 + W^(1/alpha)) minus W", ft["odds_residual"])
check_zero("net condition at the odds of phi_tilde", ft["net_at_phi_tilde"])
check_zero(
    "A_eff channel at the odds of phi_underbar", ft["a_eff_channel_at_phi_underbar"]
)
check_zero("phi_tilde at m = 0 minus phi_underbar", ft["limit_m_to_zero_residual"])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

d ln A_eff/d lambda minus the closed form: residual = 0
net condition / Pi_L minus its form in the odds: residual = 0
odds of W^(1/alpha)/(1 + W^(1/alpha)) minus W: residual = 0
net condition at the odds of phi_tilde: residual = 0
A_eff channel at the odds of phi_underbar: residual = 0
phi_tilde at m = 0 minus phi_underbar: residual = 0


The last sentence of Proposition 2(ii): $\underline\phi$ increases in $\Omega$, and $\ln\Omega = \ln q/\alpha$ with $q = (r-\mu_H)/(r-\mu_L) < 1$ under (A1). The derivatives below are positive in $\alpha$ (since $\ln q < 0$) and in $r$, negative in $\mu_H$ and positive in $\mu_L$, so $\underline\phi$ increases in $\alpha$ and $r$ and decreases in the growth premium $\mu_H - \mu_L$.

In [21]:
latex_names = {"alpha": "\\alpha", "r": "r", "mu_H": "\\mu_H", "mu_L": "\\mu_L"}
for key, expr in ft["dlnOmega"].items():
    show("\\partial\\ln\\Omega/\\partial " + latex_names[key], expr)
    check_zero(
        f"d ln Omega/d {key} minus its signed form", ft["dlnOmega_residuals"][key]
    )

<IPython.core.display.Math object>

d ln Omega/d alpha minus its signed form: residual = 0


<IPython.core.display.Math object>

d ln Omega/d r minus its signed form: residual = 0


<IPython.core.display.Math object>

d ln Omega/d mu_H minus its signed form: residual = 0


<IPython.core.display.Math object>

d ln Omega/d mu_L minus its signed form: residual = 0


Numerically, at baseline, the semi-elasticity $m$, the two thresholds and the regularity condition, against `DuopolyModel.faith_threshold` and `faith_threshold_exact`; then the sign of a centred finite difference of the code's $X_D$ in $\lambda$ on either side of $\tilde\phi$ (monopoly, fixed $K$, zero coupon), and the arrival rate $\bar\lambda$ at which $\phi^*(\lambda) = \tilde\phi(\lambda)$.

In [22]:
m_val = float(mk["m"].subs(subs_all))
phi_t = float(ft["phi_tilde"].subs(ft["m"], m_val).subs(subs_all))
phi_u = float(ft["phi_underbar"].subs(subs_all))
reg = float((p.r - p.mu_L) - m_val * p.lam * (p.r - p.mu_L + p.lam))
duo0 = DuopolyModel(p)
print(f"m = {m_val:.4f}; regularity (r - mu_L) - m lambda Delta = {reg:.4f} > 0")
print(f"Omega = {float(ft['Omega'].subs(subs_all)):.4f}")
print(f"phi_underbar = {phi_u:.4f} (code {duo0.faith_threshold():.4f})")
print(f"phi_tilde    = {phi_t:.4f} (code {duo0.faith_threshold_exact():.4f})")
print(f"phi*         = {phi_code:.4f}")


def X_D_at(lam_value, phi_value, K=1.0):
    model = DuopolyModel(p.with_param(lam=lam_value, lam_0=lam_value), leverage=0.0)
    return model.default_boundary(phi_value, K, 0.0, 0.0)


eps = 1e-4
print("\nCentred difference of X_D in lambda (shutdown boundary, monopoly, K = 1):")
for phi_value in (phi_t - 0.05, phi_t - 0.01, phi_t + 0.01, phi_t + 0.05, phi_code):
    d = (X_D_at(p.lam + eps, phi_value) - X_D_at(p.lam - eps, phi_value)) / (2 * eps)
    print(f"  phi = {phi_value:.4f}: dX_D/dlambda = {d:+.4e}")


def gap(lam_value):
    q = p.with_param(lam=lam_value, lam_0=lam_value)
    return (
        SingleFirmModel(q).closed_form_phi_star()
        - DuopolyModel(q).faith_threshold_exact()
    )


lam_bar = optimize.brentq(gap, 0.005, 0.10)
print(f"\nphi*(lambda) = phi_tilde(lambda) at lambda_bar = {lam_bar:.4f}")

m = 0.7681; regularity (r - mu_L) - m lambda Delta = 0.0939 > 0
Omega = 0.2197
phi_underbar = 0.1801 (code 0.1801)
phi_tilde    = 0.3219 (code 0.3219)
phi*         = 0.7009

Centred difference of X_D in lambda (shutdown boundary, monopoly, K = 1):
  phi = 0.2719: dX_D/dlambda = +4.6820e-03
  phi = 0.3119: dX_D/dlambda = +8.8134e-04
  phi = 0.3319: dX_D/dlambda = -8.5809e-04
  phi = 0.3719: dX_D/dlambda = -4.0920e-03
  phi = 0.7009: dX_D/dlambda = -2.6346e-02

phi*(lambda) = phi_tilde(lambda) at lambda_bar = 0.0341


Under joint re-optimization $\phi^*$ also moves with $\lambda$, but since $X_D \propto 1/A_{\text{eff}}$ and $\partial A_{\text{eff}}/\partial\phi = 0$ at $\phi^*$, the reallocation has no first-order effect on $X_D$ (an envelope argument). The smooth-pasting conditions behind the closed-form boundary are checked against the code's equity value:

In [23]:
duo = DuopolyModel(p, leverage=0.40, coupon_rate=0.05, bankruptcy_cost=0.30)
phi_test, K_test = 0.50, 1.0
beta_neg = duo._negative_root("L", p.lam)
X_D = duo.default_boundary(phi_test, K_test, 0.0, 0.0)
A_eff_val = duo._effective_revenue_coeff(phi_test, K_test, 0.0, 0.0, monopolist=True)
c_D = duo.coupon_payment(K_test)
print(f"beta_L^- from the code = {beta_neg:.6f}; from the formula = {bLn:.6f}")
X_D_formula = (
    beta_neg / (beta_neg - 1) * (c_D / p.r + p.delta * K_test / p.r) / A_eff_val
)
print(f"X_D from the code = {X_D:.8f}; from the formula = {X_D_formula:.8f}")
V_XD = A_eff_val * X_D - p.delta * K_test / p.r
E_prime = A_eff_val + beta_neg * (c_D / p.r - V_XD) / X_D
print(f"Smooth pasting E'(X_D) = {E_prime:.2e}")
h_fd = 1e-7 * X_D
E_up = duo.equity_value(X_D + h_fd, phi_test, K_test, 0.0, 0.0) + (
    1 - duo.leverage
) * duo.investment_cost(K_test)
print(
    f"Going-concern equity just above X_D (finite difference slope) = {E_up / h_fd:.2e}"
)

beta_L^- from the code = -2.334995; from the formula = -2.334995
X_D from the code = 0.03031392; from the formula = 0.03031392
Smooth pasting E'(X_D) = 1.78e-15
Going-concern equity just above X_D (finite difference slope) = 1.61e-06


*Established:* the two channels, the markup semi-elasticity $m$, the linear reduction in the odds, the thresholds $\underline\phi$ and $\tilde\phi$ and the limit $\tilde\phi \to \underline\phi$ as $m \to 0$ (symbolic); the baseline values, the sign flip at $\tilde\phi$ of the code's boundary, $\bar\lambda$, and the smooth-pasting conditions (numerical). All of this is under the single-boundary reduction of `@eq-default-boundary`; the coupled boundary that the preemption game uses is not covered.

## 7. The recovery rule (`@eq-recovery`)

Creditors seize the inference business net of the operating cost of its capacity, $\Lambda(X) = [(1-\phi)K]^\alpha X/\Delta - \delta(1-\phi)K/r$ (monopoly or symmetric shares). Three identities: installed value is exactly $\Lambda$ plus the training business; at $\phi^*$, $\Lambda = (1-\phi^*)V_L$ (the inference-share identity of Section 5); and at the single-boundary $X_D$, where $A_{\text{eff}}X_D = M(C_D + \delta K)/r$, $\Lambda(X_D) < 0$ iff $C_D < \delta K(1-M)/M$.

In [24]:
rec = sd.recovery_split(syms)
show("\\Lambda(X)", rec["Lambda"])
check_zero("V_L - Lambda - training business", rec["split_residual"])
show("\\Lambda(X_D) \\text{ at } \\phi^*", rec["Lambda_at_X_D"])
check_zero(
    "Lambda(X_D) - (1 - phi) M/r (C_D - delta K (1 - M)/M)", rec["sign_residual"]
)

M_val = bLn / (bLn - 1)
bound = (1 - M_val) / M_val
print(f"\nM = {M_val:.4f}; Lambda(X_D) < 0 unless C_D > {bound:.4f} delta K")
duo_rec = DuopolyModel(p, coupon_rate=0.05, bankruptcy_cost=0.30)
print("Precommitted-scale policy, par debt at r_c = 0.05:")
for lev in (0.20, 0.40, 0.70):
    C = duo_rec.coupon_payment(K_s, lev)
    R = duo_rec.recovery_value(opt.phi, K_s, 0.0, 0.0, lev)
    ratio_C = C / (p.delta * K_s)
    print(f"  leverage {lev:.2f}: C_D/(delta K) = {ratio_C:.3f}; recovery = {R:.3e}")

<IPython.core.display.Math object>

V_L - Lambda - training business: residual = 0


<IPython.core.display.Math object>

Lambda(X_D) - (1 - phi) M/r (C_D - delta K (1 - M)/M): residual = 0

M = 0.7001; Lambda(X_D) < 0 unless C_D > 0.4283 delta K
Precommitted-scale policy, par debt at r_c = 0.05:
  leverage 0.20: C_D/(delta K) = 0.042; recovery = 0.000e+00
  leverage 0.40: C_D/(delta K) = 0.084; recovery = 0.000e+00
  leverage 0.70: C_D/(delta K) = 0.147; recovery = 0.000e+00


*Established:* the split of installed value, and the sign condition for $\Lambda(X_D)$ at the optimal allocation under the single-boundary reduction (symbolic); the threshold $(1-M)/M$ at baseline and zero recovery at the precommitted-scale policy for the leverages shown (numerical). The caps of `@eq-recovery` at $U(X_D)$ and at $C_D/r$ are definitions and are not checked. In the preemption game recovery is computed with the coupled boundary, which is not covered here.

## 8. The contest and the follower (Proposition 3(i) and (ii))

### 8.1 The Tullock derivative and the allocation at a common $\phi$

For $f(u) = u^{2\alpha}/(u^\alpha + \bar u^\alpha)$, $f'(u) = \alpha u^{\alpha-1}s(2-s)$ with $s = u^\alpha/(u^\alpha + \bar u^\alpha)$, and $s$ is invariant to scaling both capacities. When both firms have the same $\phi$, the capacity ratio, hence $s$, is the same in the two regimes, and by the chain rule the entrant's allocation derivative is the single firm's times $s(2-s)$. Below $\psi$ stands for $1-\phi$, both positive.

In [25]:
tl = sd.tullock_factorization(syms)
show("f'(u)", tl["f_prime"])
check_zero("f'(u) - alpha u^(alpha-1) s (2 - s)", tl["derivative_residual"])
check_zero("s(t u, t ubar) - s(u, ubar)", tl["scale_invariance_residual"])
show(
    "\\partial_{\\phi_i}\\,[w_L f((1-\\phi_i)K_i) + w_H f(\\phi_i K_i)]",
    tl["chain_rule"],
)
show(
    "\\partial A_{\\text{eff},i}/\\partial\\phi_i \\text{ at a common } \\phi",
    tl["common_allocation_derivative"],
)
check_zero(
    "allocation derivative at a common phi minus the factored form",
    tl["common_allocation_residual"],
)

<IPython.core.display.Math object>

f'(u) - alpha u^(alpha-1) s (2 - s): residual = 0
s(t u, t ubar) - s(u, ubar): residual = 0


<IPython.core.display.Math object>

<IPython.core.display.Math object>

allocation derivative at a common phi minus the factored form: residual = 0


The bracket is the single firm's condition of Section 5, so a common allocation is a critical point iff it is $\phi^*$, whatever the two capacities. *Established:* exactly that critical-point statement (symbolic). Not established: that the critical point is a maximum, or anything about the leader's allocation in the solved game, where the rival is not at the leader's allocation; the paper reports the leader's $\phi_L$ as a numerical finding.

### 8.2 The follower after the switch (`@eq-follower-foc`)

Against a leader with training capacity $T_L$, the follower's revenue coefficient is $a(K) = A_HK^{2\alpha}/(K^\alpha + T_L^\alpha)$, so $d\ln a/d\ln K = \alpha(2 - s_F)$, and the log-derivative of $a^{\beta_H}/\Theta^{\beta_H-1}$ in $\ln K$ is $\beta_H\alpha(2-s_F) - (\beta_H - 1)K\Theta'/\Theta$. Setting it to zero is `@eq-follower-foc`. Since $s_F$ rises in $K$ and $K\Theta'/\Theta$ rises from $1$ to $\gamma$, the derivative is strictly decreasing, positive near $K = 0$ iff $2\alpha\beta_H > \beta_H - 1$ and negative for large $K$ iff $\alpha\beta_H < \gamma(\beta_H - 1)$.

In [26]:
fo = sd.follower_capacity_condition(syms)
check_zero("d ln a/d ln K - alpha (2 - s_F)", fo["elasticity_a_residual"])
show("K\\Theta'/\\Theta", fo["elasticity_theta"])
print(
    "Limits of K Theta'/Theta at gamma = 3/2 (K -> 0, K -> oo):",
    fo["theta_elasticity_limits_at_gamma_3_2"],
)
ipy_display(fo["foc"])
check_zero("eq-follower-foc / Theta minus the log-derivative", fo["foc_residual"])
show("X_F^*", fo["trigger"])
small_ok = 2 * p.alpha * p.beta_H > p.beta_H - 1
large_ok = p.alpha * p.beta_H < p.gamma * (p.beta_H - 1)
print(f"\nAt baseline: 2 alpha beta_H > beta_H - 1: {small_ok}")
print(f"alpha beta_H < gamma (beta_H - 1): {large_ok}")

d ln a/d ln K - alpha (2 - s_F): residual = 0


<IPython.core.display.Math object>

Limits of K Theta'/Theta at gamma = 3/2 (K -> 0, K -> oo): (1, 3/2)


      ⎛      α        ⎞                            ⎛     γ    ⎞
      ⎜     K         ⎟ ⎛K⋅δ    γ  ⎞               ⎜δ   K ⋅c⋅γ⎟
α⋅β_H⋅⎜- ───────── + 2⎟⋅⎜─── + K ⋅c⎟ = K⋅(β_H - 1)⋅⎜─ + ──────⎟
      ⎜   α      α    ⎟ ⎝ r        ⎠               ⎝r     K   ⎠
      ⎝  K  + T_L     ⎠                                        

eq-follower-foc / Theta minus the log-derivative: residual = 0


<IPython.core.display.Math object>


At baseline: 2 alpha beta_H > beta_H - 1: True
alpha beta_H < gamma (beta_H - 1): True


Numerically, the root of `@eq-follower-foc` and the trigger $X_F^*$, solved here from the symbolic expression, against `ExactPreemptionGame.post_switch_entry` (the follower's claim without the option to stop), for three values of the leader's training capacity chosen as inputs. With $T_L = 0$ the follower is the single firm after the switch and the root is $K_H^*$.

In [27]:
game = ExactPreemptionGame(p, scale="envelope")
subs_f = dict(subs_all)
subs_f[fo["A_H"]] = p.A_H
for T_L in (0.0, 1e-3, 1e-2):
    subs_f[fo["T_L"]] = T_L
    log_derivative = sp.lambdify(syms["K"], fo["log_derivative"].subs(subs_f))

    def in_log_K(t, fn=log_derivative):
        return fn(np.exp(t))

    K_F = float(np.exp(optimize.brentq(in_log_K, -30.0, 10.0)))
    X_F = float(fo["trigger"].subs(subs_f).subs(syms["K"], K_F))
    entry = game.post_switch_entry(T_L)
    print(f"T_L = {T_L:.0e}: K_F = {K_F:.8f} (game {entry.K_F:.8f})")
    print(f"           X_F* = {X_F:.8f} (game {entry.X_F:.8f})")

T_L = 0e+00: K_F = 0.00672664 (game 0.00672664)
           X_F* = 0.00278296 (game 0.00278296)
T_L = 1e-03: K_F = 0.09825334 (game 0.09825334)
           X_F* = 0.02736831 (game 0.02736831)
T_L = 1e-02: K_F = 0.32376312 (game 0.32376312)
           X_F* = 0.08761833 (game 0.08761833)


*Established:* the elasticity $\alpha(2 - s_F)$, the equivalence of the log-derivative with `@eq-follower-foc`, the monotonicity ingredients and the two uniqueness conditions (symbolic, with the limits of $K\Theta'/\Theta$ at $\gamma = 3/2$); agreement of the root with the game's solver (numerical). Not established: the follower's policy with the option to stop, its pre-switch entry decision, or any equilibrium level of the game.

## 9. The Green's function of the L-regime operator (`@eq-green`)

Every claim held before the switch solves $\tfrac12\sigma^2X^2V'' + \mu_LXV' - (r+\lambda)V + f(X) = 0$ with $f$ its flow plus $\lambda$ times its value at the switch. The bounded solution without the explosive root is
$$P(X) = \frac{2}{\sigma^2(\beta_L^+ - \beta_L^-)}\Big[X^{\beta_L^-}\!\int_0^X f(y)y^{-\beta_L^- - 1}dy + X^{\beta_L^+}\!\int_X^\infty f(y)y^{-\beta_L^+ - 1}dy\Big].$$
For a generic $f$, Leibniz's rule and the root conditions $Q_L(\beta_L^\pm) = 0$ give a zero residual:

In [28]:
gr = sd.green_function_residual(syms)
show("P(X)", gr["P"])
check_zero("ODE residual of P for generic f", gr["residual"])

<IPython.core.display.Math object>

ODE residual of P for generic f: residual = 0


Numerically, `ForcedClaim` (the implementation of `@eq-green` in `preemption_exact.py`) with the piecewise forcing $\lambda F_H$ of Section 4 reproduces the value $S$ of never investing before the switch, computed in `piecewise_option.py` from the two-region closed form:

In [29]:
claim = ForcedClaim(
    p,
    piece.X_H,
    low=((p.lam * piece.B_H, p.beta_H),),
    high=((p.lam * piece.a_H, 1.0), (-p.lam * piece.b_H, 0.0)),
)
for ratio in (0.2, 0.9, 1.5, 5.0, 50.0):
    x = ratio * piece.X_H
    green, closed = float(claim(x)[0]), piece.pure_switching_value(x)
    print(f"X = {ratio:5.1f} X_H*: Green {green:.10e}, closed form S {closed:.10e}")

X =   0.2 X_H*: Green 1.8498364027e-04, closed form S 1.8498364027e-04
X =   0.9 X_H*: Green 1.7608419344e-03, closed form S 1.7608419344e-03
X =   1.5 X_H*: Green 3.4921383361e-03, closed form S 3.4921383361e-03
X =   5.0 X_H*: Green 1.3922154419e-02, closed form S 1.3922154419e-02
X =  50.0 X_H*: Green 1.4834201478e-01, closed form S 1.4834201478e-01


*Established:* that `@eq-green` solves the L-regime equation for any forcing for which the integrals converge (symbolic; convergence assumed), and that the code's implementation agrees with the closed form for the piecewise forcing of the single firm (numerical). The claims of the levered game built on it are not checked here.

## 10. Parameter admissibility

(A2) is $1/\gamma < (\beta_H - 1)/(\alpha\beta_H) < 1$. At the baseline values of the other parameters it is a window for $\alpha$, $(\beta_H-1)/\beta_H < \alpha < \gamma(\beta_H-1)/\beta_H$. The L-regime insufficiency condition is $\alpha \leq 1 - 1/\beta_L^+$. Both depend on $r$ through the roots, so (A2) is also checked at the archetype WACCs.

In [30]:
alpha_lo = (p.beta_H - 1) / p.beta_H
alpha_hi = p.gamma * (p.beta_H - 1) / p.beta_H
alpha_ins = 1 - 1 / p.beta_L
print(f"(A2) window for alpha at baseline: ({alpha_lo:.3f}, {alpha_hi:.3f})")
print(f"L-regime insufficiency holds for alpha <= {alpha_ins:.3f}")
ratio_A2 = (p.beta_H - 1) / (p.alpha * p.beta_H)
print(f"(A2) ratio at baseline: {ratio_A2:.4f}; 1/gamma = {1 / p.gamma:.4f}")

print(f"\n{'r':>6s}  {'beta_H':>8s}  {'ratio':>8s}  {'(A2)':>6s}")
for r_val in (0.10, 0.12, 0.14, 0.15, 0.18):
    q = p.with_param(r=r_val)
    ratio_r = (q.beta_H - 1) / (q.alpha * q.beta_H)
    verdict = "holds" if 1 / q.gamma < ratio_r < 1 else "fails"
    print(f"{r_val:6.2f}  {q.beta_H:8.4f}  {ratio_r:8.4f}  {verdict:>6s}")

(A2) window for alpha at baseline: (0.356, 0.534)
L-regime insufficiency holds for alpha <= 0.668
(A2) ratio at baseline: 0.8901; 1/gamma = 0.6667

     r    beta_H     ratio    (A2)
  0.10    1.3871    0.6976   holds
  0.12    1.5529    0.8901   holds
  0.14    1.7060    1.0346   fails
  0.15    1.7787    1.0944   fails
  0.18    1.9837    1.2397   fails


*Established:* the (A2) window and the insufficiency bound at baseline, and the (A2) status at the five archetype WACCs (numerical). Over the admissible sensitivity range of $\alpha$ the insufficiency condition holds because the (A2) window lies below its bound.

## 11. Summary

| Section | Paper result | What the notebook establishes |
|:--|:--|:--|
| 2 | `@eq-beta-H`, the L-regime roots, $Q_L(\beta_H) < 0$ | Root formulas; $Q_L(\beta_H) = (\mu_L - \mu_H)\beta_H - \lambda$ (symbolic); ordering at baseline (numerical) |
| 3 | Proposition 1(ii), `@eq-trigger-H`, `@eq-foc-K`, `@eq-K-H` | Trigger, closed-form capacity as the root of the first-order condition, independence from the revenue coefficient, exponents behind (A2) (symbolic); agreement with the code |
| 4.1, 4.2 | `@eq-particular-C`, `@eq-option-L-general`, `@eq-option-L-upper` | Particular solutions on both sides of $X_H^*$ (symbolic) |
| 4.3, 4.4 | Proposition 1(iii), `@eq-scale-foc`, `@eq-premium` (Step 7) | Linear structure of the two-region system; the premium representation and smooth pasting as its first-order condition (symbolic); the code's optimum satisfies the conditions; $A_w \approx -1.18 \times 10^4$, $A_1 \approx -0.85 \times 10^4$, $-1.01 \times 10^4$ at $K_H^*$, $S$ about 10\% below $\Gamma X^{\beta_H}$ at $X_H^*$ (numerical) |
| 4.5 | `@eq-l-insufficiency` | $\Phi_L \approx 1.67$ at baseline; the condition does not imply $A_1 = 0$, and $A_1 < 0$ in the exact solution |
| 4.6 | `@eq-trigger-phi`, the pure-power approximation | Trigger and smooth-fit coefficient; the HJB residual is $Q_L(\beta_H)(B - \Gamma)X^{\beta_H}$ (symbolic); $B \approx 16.5$ against $\Gamma \approx 21.1$ (numerical) |
| 4.7 | `@eq-waiting-margin` | Ratio of the leading asymptotic terms (symbolic); value $1.08$ at baseline, as in the code. The criterion itself is not proved |
| 5 | Proposition 1(i), `@eq-a-eff`, `@eq-phi-star` | Closed form, concavity, inference-share identity, comparative statics (symbolic); Inada limits, agreement with the code |
| 6 | Proposition 2, `@eq-default-boundary`, `@eq-phi-underbar`, `@eq-phi-tilde` | Both channels of $\partial X_D/\partial\lambda$, $m$, the linear reduction, $\underline\phi$, $\tilde\phi$, the $m \to 0$ limit, the comparative statics of $\underline\phi$ (symbolic); $m \approx 0.77$, $\underline\phi \approx 0.18$, $\tilde\phi \approx 0.32$, the sign flip at $\tilde\phi$, $\bar\lambda \approx 0.034$, smooth pasting (numerical) |
| 7 | `@eq-recovery` | Split of installed value; sign of $\Lambda(X_D)$ at $\phi^*$ and the threshold $(1-M)/M \approx 0.43$ (symbolic and numerical) |
| 8 | Proposition 3(i), `@eq-follower-foc`; Proposition 3(ii), first statement | Follower elasticity and first-order condition, uniqueness conditions; Tullock derivative, scale invariance, critical-point factorization (symbolic); the root matches the game's solver |
| 9 | `@eq-green` | Residual zero for generic forcing (symbolic); the code's Green's function reproduces the value of waiting (numerical) |
| 10 | Assumption 1, Internet Appendix B boundary analysis | (A2) window $(0.356, 0.534)$ for $\alpha$, insufficiency bound $0.67$, (A2) at archetype WACCs |

The baseline numbers in the table are those the cells above compute from the model code.

## 12. Completeness audit

**Corrections in this version (issue #185, review comment C3).**

1. The statement that the L-regime insufficiency condition (the former Assumption A3) implies $A_1 = 0$, and the printed conclusion that the single power $F_L = CX^{\beta_H}$ is valid at baseline, are withdrawn. Section 4.5 states what the condition says, and Sections 4.3 and 4.4 solve the piecewise problem, in which $A_1 < 0$.
2. The option coefficient of the approximation is the smooth-fit $B \approx 16.5$, not $\Gamma \approx 21.1$ (Section 4.6); the earlier numerical cell that matched $F_L$ against $\Gamma X^{\beta_H}$ checked the simple-mode code path against its own formula and is removed.
3. The faith-based survival argument now includes the markup channel and the exact threshold $\tilde\phi$ (Section 6); the earlier chain-rule argument through $A_{\text{eff}}$ alone stated a sufficient condition that is not sufficient.
4. The admissible range of $\alpha$ is the (A2) window $(0.356, 0.534)$ at baseline, not $[0.20, 0.60]$ (Section 10).
5. New sections cover Proposition 3(i) and the first statement of (ii), $\tilde\phi$, the piecewise problem, the waiting margin, the recovery rule and the Green's function.

**Covered by Lean, not by this notebook.** The general ordering of the characteristic roots, the closed forms and comparative statics of Propositions 1 and 2, and the contest-share lemmas (`lean/README.md`).

**Not covered (numerical results, or outside the symbolic scope).**

- Optimality of the candidate value functions; the HJB derivation itself.
- Uniqueness of the capacity maximum in Proposition 1(iii) (a grid finding in `piecewise_option.py`).
- The waiting-margin criterion as a criterion for the existence of a pre-switch trigger, and the waiting result at baseline (computed in `piecewise_option.py` and `capability_stock.py`).
- The levels and error of the pure-power approximation (Internet Appendix B).
- Proposition 3(iii) to (v) and Remark 4: existence of the rent-equalization trigger, the trigger of an assigned leader and the order of the two, which of them dates the first investment, the leader's allocation, scale and distance to default, the spread at entry, and leverage effects (all numerical, `preemption_exact.py`, `preemption_levered.py`).
- The coupled regime-switching default boundary and the levered claims of the game; the caps of the recovery rule.
- Dario's dilemma (Numerical Finding 1).
- **Pending issue #181:** the concavity of the installed value that the testable predictions will use. It is not derived here and will be added once #181 states it.